In [1]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

import os
from dotenv import load_dotenv
load_dotenv()


from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

model = 'openai/gpt-oss-20b'
documents = load_faq_data()
index = build_index(documents)


In [2]:
assistant = RAGBase(
    index, 
    openai_client, 
    model=model
)

response = assistant.rag('How do I run Ollama locally?')
print(response)

**Running Ollama locally**

1. **Install Ollama**  
   - **macOS**: Download the `.pkg` from <https://ollama.com/download> and run it.  
   - **Windows**: Download the `.msi` from <https://ollama.com/download> and install it.  
   - **Linux**: In a terminal execute  
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**  
   Open a terminal and run:  
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model (~4 GB) and launches a chat‑like interface.

3. **Verify the local server**  
   ```bash
   curl http://localhost:11434
   ```
   A JSON response such as `{"models": [...]} ` indicates the server is running.

4. **Use it from Python (optional)**  
   ```bash
   pip install ollama
   ```
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "your_prompt"}]
   )
   print(response['message']['content'])
   ```

That’s all you need to run Ollama locally on a

In [3]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I’m sorry, but the provided FAQ context does not contain any information about running Olama locally.


In [4]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='openai/gpt-oss-20b',
    input=messages,
)

response.output_text

'Absolutely! 🚀\n\nHere’s what you can do to get started:\n\n1. **Identify the Course**  \n   - If you have a course code, title, or a link, paste it here so I can pull up the exact details (availability, prerequisites, schedule, etc.).\n\n2. **Check Availability**  \n   - Once I have the specifics, I’ll let you know if seats are still open and if you’re eligible based on prerequisites.\n\n3. **Enroll**  \n   - If spots are available, I’ll walk you through the enrollment steps—usually a quick sign‑up via the learning platform or a confirmation email from the instructor.\n\n4. **Prepare**  \n   - I can also share any recommended pre‑reading or materials that will help you hit the ground running.\n\nJust let me know the course name or code, and we’ll get you on board!'

In [5]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [7]:
response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
    tools=[search_tool]
)

In [8]:
response.output_text

''

In [9]:
len(response.output)

2

In [10]:
response

Response(id='resp_01kvqf2grmfgracvg481e0gv8j', created_at=1782125249.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='qwen/qwen3-32b', object='response', output=[ResponseReasoningItem(id='resp_01kvqf2grmfgrradhgs4dcw3yb', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to figure out what they need.\n\nFirst, the question is about joining the course, so it\'s likely about enrollment. The provided function is to search the FAQ. I should check if there\'s an existing FAQ entry that addresses joining or enrollment deadlines.\n\nThe function requires a query parameter. The user\'s question is about joining, so the query should be something like "Can I join the course?" or "Enrollment deadline". \n\nI should call the search function with the query "joining the course" to see if there\'s relevant information in the FAQ. That should help determine if there\'s an answer 

In [11]:
print(response.output)

[ResponseReasoningItem(id='resp_01kvqf2grmfgrradhgs4dcw3yb', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to figure out what they need.\n\nFirst, the question is about joining the course, so it\'s likely about enrollment. The provided function is to search the FAQ. I should check if there\'s an existing FAQ entry that addresses joining or enrollment deadlines.\n\nThe function requires a query parameter. The user\'s question is about joining, so the query should be something like "Can I join the course?" or "Enrollment deadline". \n\nI should call the search function with the query "joining the course" to see if there\'s relevant information in the FAQ. That should help determine if there\'s an answer already available, like enrollment dates or how to join.\n', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"joining the co

In [12]:
call = response.output[0]

In [13]:
call

ResponseReasoningItem(id='resp_01kvqf2grmfgrradhgs4dcw3yb', summary=[], type='reasoning', content=[Content(text='Okay, the user is asking if they can join the course since they just discovered it. I need to figure out what they need.\n\nFirst, the question is about joining the course, so it\'s likely about enrollment. The provided function is to search the FAQ. I should check if there\'s an existing FAQ entry that addresses joining or enrollment deadlines.\n\nThe function requires a query parameter. The user\'s question is about joining, so the query should be something like "Can I join the course?" or "Enrollment deadline". \n\nI should call the search function with the query "joining the course" to see if there\'s relevant information in the FAQ. That should help determine if there\'s an answer already available, like enrollment dates or how to join.\n', type='reasoning_text')], encrypted_content=None, status='completed')

In [14]:
import json

# Get the function call
call = next(item for item in response.output if item.type == "function_call")

print(call.name)        # search
print(call.arguments)   # {"query":"joining the course"}

args = json.loads(call.arguments)
print(args)
print(args["query"])

search
{"query":"joining the course"}
{'query': 'joining the course'}
joining the course


In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
result_json

'[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\\n\\nA typical workflow is:\\n\\n1. Watch the lesson videos.\\n2. Work through the lesson notebooks/code.\\n3. Read the homework instructions on GitHub.\\n4. Submit answers through the course platform before the deadline.\\n\\nHomework is similar to the lesson flow, but uses a different dataset or slightly diff

In [18]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [19]:
messages.append(call)

In [20]:
messages.append(function_call_output)

In [21]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"joining the course"}', call_id='q5h0khdm2', name='search', type='function_call', id='q5h0khdm2', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'q5h0khdm2',
  'output': '[\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/ll

In [22]:
response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
    tools=[search_tool]
)

In [23]:
print(response.output_text)

You can join the course at any time! However, if you'd like to receive a certificate, you'll need to submit your project while the course is still accepting submissions. The course materials, videos, and deadlines are available through the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [24]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(895, 252)

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [ ]:
response = openai_client.responses.create(
    model='qwen/qwen3-32b',
    input=messages,
    tools=[search_tool]
)


In [28]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
You can join the course at any time! However, if you'd like to receive a certificate, you'll need to submit your project while the course is still accepting submissions. The course materials, videos, and deadlines are available through the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='resp_01kvqf341ef8q9vpvrp24fychc', summary=[], type='reasoning', content=[Content(text='Okay, the user asked, "I just discovered the course. Can I join it?" Let me look at the tool response.\n\nThe tool response includes a list of search results. The last entry (id: 74eb249bbf) has the exact question the user is asking. The answer there says, "Yes, but if you want to receive a certifi

In [31]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='qwen/qwen3-32b',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break


iteration #1...
function_call: search {"query":"joining the course late"}
iteration #2...
ASSISTANT:
The answer to your question is: **Yes, you can still join the course.** However, if you want to receive a certificate, you must submit your project while we’re still accepting submissions. For details on deadlines and requirements, check the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).


In [35]:
def agent_loop(instructions, question, model='qwen/qwen3-32b') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform    more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [37]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course"}
iteration #2...
ASSISTANT:
You can join the course! However, if you'd like to receive a certificate, you'll need to submit your project while the course is still accepting submissions. 

Additional relevant info from the FAQ:
- The course will be offered again in **Summer 2027**.
- You can run the coursework locally if preferred (setup instructions available).
- Free/low-cost alternatives to OpenAI API are allowed for course exercises.

Would you like clarification on any of these points?
